# Appendix C: LFM3 model source

The trainable candidate is [LiquidAI/LFM3-1.7B-Base](https://huggingface.co/LiquidAI/LFM3-1.7B-Base). It combines gated causal convolutions, sliding attention and global attention. The code below is the same PyTorch implementation used throughout this edition. Weights use the separate LFM Open License v1.0; model code is Apache-2.0.

In [ ]:
import torch
from torch import nn
from torch.nn import functional as F

In [ ]:
LFM3_CONFIG_1_7B = {
    "vocab_size": 128_000,
    "context_length": 262_144,
    "emb_dim": 2048,
    "n_heads": 16,
    "n_layers": 24,
    "hidden_dim": 7680,
    "head_dim": 128,
    "n_kv_groups": 4,
    "conv_kernel_size": 7,
    "sliding_window": 256,
    "norm_eps": 1e-5,
    "dtype": torch.bfloat16,
    "layer_types": (
        "conv",
        "conv",
        "full_attention",
        "conv",
        "sliding_attention",
        "conv",
        "full_attention",
        "conv",
        "sliding_attention",
        "conv",
        "full_attention",
        "conv",
        "sliding_attention",
        "conv",
        "full_attention",
        "conv",
        "sliding_attention",
        "conv",
        "full_attention",
        "conv",
        "sliding_attention",
        "conv",
        "full_attention",
        "conv",
    ),
}

## KVCache

In [ ]:
class KVCache:
    """Store attention (K, V) or a convolution input window per layer."""

    def __init__(self, n_layers):
        self.cache = [None] * n_layers
        self.position = 0

    def get(self, layer_idx):
        return self.cache[layer_idx]

    def update(self, layer_idx, value):
        self.cache[layer_idx] = value

    def get_all(self):
        return self.cache

    def reset(self):
        self.cache = [None] * len(self.cache)
        self.position = 0

## RMSNorm

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps, dtype):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim, dtype=dtype))
        self.eps = eps

    def forward(self, x):
        normalized = x.float() * torch.rsqrt(
            x.float().square().mean(-1, keepdim=True) + self.eps
        )
        return self.weight * normalized.to(x.dtype)

## FeedForward

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        dim, hidden, dtype = cfg["emb_dim"], cfg["hidden_dim"], cfg["dtype"]
        self.w1 = nn.Linear(dim, hidden, bias=False, dtype=dtype)
        self.w3 = nn.Linear(dim, hidden, bias=False, dtype=dtype)
        self.w2 = nn.Linear(hidden, dim, bias=False, dtype=dtype)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

## ShortConv

In [ ]:
class ShortConv(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        dim, dtype = cfg["emb_dim"], cfg["dtype"]
        self.width = cfg["conv_kernel_size"]
        self.conv = nn.Conv1d(
            dim, dim, self.width, groups=dim, bias=False, dtype=dtype
        )
        self.in_proj = nn.Linear(dim, 3 * dim, bias=False, dtype=dtype)
        self.out_proj = nn.Linear(dim, dim, bias=False, dtype=dtype)

    def forward(self, x, state=None, mask=None):
        if mask is not None:
            x = x * mask.unsqueeze(-1).to(x.dtype)
        b, c, values = self.in_proj(x).transpose(1, 2).chunk(3, dim=1)
        values = b * values
        if state is None:
            history = F.pad(values, (self.width - 1, 0))
        else:
            history = torch.cat((state, values), dim=-1)
        out = self.conv(history)
        next_state = (
            history[..., -(self.width - 1) :]
            if self.width > 1
            else history[..., :0]
        )
        return self.out_proj((c * out).transpose(1, 2)), next_state

## Attention

In [ ]:
class Attention(nn.Module):
    def __init__(self, cfg, layer_type):
        super().__init__()
        dim, dtype = cfg["emb_dim"], cfg["dtype"]
        self.n_heads = cfg["n_heads"]
        self.n_kv = cfg["n_kv_groups"]
        self.head_dim = cfg["head_dim"]
        self.window = (
            cfg["sliding_window"]
            if layer_type == "sliding_attention"
            else None
        )
        self.q_proj = nn.Linear(
            dim, self.n_heads * self.head_dim, bias=False, dtype=dtype
        )
        self.k_proj = nn.Linear(
            dim, self.n_kv * self.head_dim, bias=False, dtype=dtype
        )
        self.v_proj = nn.Linear(
            dim, self.n_kv * self.head_dim, bias=False, dtype=dtype
        )
        self.o_proj = nn.Linear(
            self.n_heads * self.head_dim, dim, bias=False, dtype=dtype
        )
        self.gate_proj = nn.Linear(
            dim, self.n_heads, bias=False, dtype=dtype
        )
        self.q_norm = RMSNorm(self.head_dim, cfg["norm_eps"], dtype)
        self.k_norm = RMSNorm(self.head_dim, cfg["norm_eps"], dtype)
        # Global layers rotate half the dimensions, retaining the full-head
        # frequency progression; the remaining frequencies are exactly zero.
        rotated = self.head_dim if self.window else self.head_dim // 2
        base = 10_000.0 if self.window else 1_000_000.0
        self.rotated_dim = rotated
        self.rope_base = base

    def _rotate(self, x, positions):
        # Compute in FP32 on the actual device, including after HF meta loading
        # or dtype conversion. No uninitialized nonpersistent buffers survive.
        freq = self.rope_base ** (
            -torch.arange(
                0, self.rotated_dim, 2, device=x.device, dtype=torch.float32
            )
            / self.head_dim
        )
        freq = F.pad(freq, (0, self.head_dim // 2 - freq.numel()))
        angles = positions.float().unsqueeze(-1) * freq
        angles = torch.cat((angles, angles), dim=-1).unsqueeze(1)
        cos, sin = (
            angles.cos().to(x.dtype).float(),
            angles.sin().to(x.dtype).float(),
        )
        half = self.head_dim // 2
        rotated = torch.cat((-x[..., half:], x[..., :half]), dim=-1)
        return (x.float() * cos + rotated.float() * sin).to(x.dtype)

    def forward(self, x, positions, start, state=None, mask=None):
        batch, length, _ = x.shape
        q = self.q_norm(
            self.q_proj(x).view(batch, length, self.n_heads, self.head_dim)
        ).transpose(1, 2)
        k = self.k_norm(
            self.k_proj(x).view(batch, length, self.n_kv, self.head_dim)
        ).transpose(1, 2)
        v = (
            self.v_proj(x)
            .view(batch, length, self.n_kv, self.head_dim)
            .transpose(1, 2)
        )
        q, k = self._rotate(q, positions), self._rotate(k, positions)
        if state is not None:
            k, v = (
                torch.cat((state[0], k), dim=2),
                torch.cat((state[1], v), dim=2),
            )
        next_state = (k, v)
        if self.window is not None:
            next_state = (k[:, :, -self.window :], v[:, :, -self.window :])
        # Cached sliding layers keep a suffix; absolute positions still advance.
        key_start = start + length - k.shape[2]
        query_pos = torch.arange(start, start + length, device=x.device)
        key_pos = torch.arange(key_start, start + length, device=x.device)
        allowed = key_pos[None, :] <= query_pos[:, None]
        if self.window is not None:
            allowed = allowed & (
                key_pos[None, :] > query_pos[:, None] - self.window
            )
        allowed = allowed[None, None]
        if mask is not None:
            allowed = (
                allowed
                & mask[:, None, None, key_start : start + length].bool()
            )
        k, v = (
            k.repeat_interleave(self.n_heads // self.n_kv, 1),
            v.repeat_interleave(self.n_heads // self.n_kv, 1),
        )
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed)
        gate = self.gate_proj(x).sigmoid().unsqueeze(-1)
        out = (out.transpose(1, 2) * gate).reshape(batch, length, -1)
        return self.o_proj(out), next_state

## DecoderLayer

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, cfg, layer_type):
        super().__init__()
        self.layer_type = layer_type
        if layer_type == "conv":
            self.conv = ShortConv(cfg)
        else:
            self.self_attn = Attention(cfg, layer_type)
        self.feed_forward = FeedForward(cfg)
        for name in (
            "operator_norm",
            "ffn_norm",
            "post_operator_norm",
            "post_ffn_norm",
        ):
            setattr(
                self,
                name,
                RMSNorm(cfg["emb_dim"], cfg["norm_eps"], cfg["dtype"]),
            )

    def forward(self, x, positions, start, state=None, mask=None):
        normalized = self.operator_norm(x)
        if self.layer_type == "conv":
            current_mask = (
                mask[:, -x.shape[1] :] if mask is not None else None
            )
            out, state = self.conv(normalized, state, current_mask)
        else:
            out, state = self.self_attn(
                normalized, positions, start, state, mask
            )
        x = x + self.post_operator_norm(out)
        return x + self.post_ffn_norm(
            self.feed_forward(self.ffn_norm(x))
        ), state

## LFM3Model

In [ ]:
class LFM3Model(nn.Module):
    def __init__(self, cfg=None):
        super().__init__()
        self.cfg = dict(LFM3_CONFIG_1_7B if cfg is None else cfg)
        cfg = self.cfg
        if len(cfg["layer_types"]) != cfg["n_layers"]:
            raise ValueError("layer_types must contain one entry per layer")
        if cfg["n_heads"] % cfg["n_kv_groups"] or cfg["head_dim"] % 4:
            raise ValueError("Invalid attention head configuration")
        self.model = nn.Module()
        self.model.embed_tokens = nn.Embedding(
            cfg["vocab_size"], cfg["emb_dim"], dtype=cfg["dtype"]
        )
        self.model.layers = nn.ModuleList(
            [DecoderLayer(cfg, kind) for kind in cfg["layer_types"]]
        )
        self.model.embedding_norm = RMSNorm(
            cfg["emb_dim"], cfg["norm_eps"], cfg["dtype"]
        )
        self.lm_head = nn.Linear(
            cfg["emb_dim"],
            cfg["vocab_size"],
            bias=False,
            dtype=cfg["dtype"],
        )
        self.lm_head.weight = self.model.embed_tokens.weight

    @property
    def tok_emb(self):
        return self.model.embed_tokens

    @property
    def trf_blocks(self):
        return self.model.layers

    def reset_kv_cache(self):
        # Positions and recurrent state belong to the cache, not the model.
        pass

    def forward(self, in_idx, cache=None, attn_mask=None):
        start = cache.position if cache is not None else 0
        end = start + in_idx.shape[1]
        if end > self.cfg["context_length"]:
            raise ValueError(
                f"Sequence length {end} exceeds context length "
                f"{self.cfg['context_length']}"
            )
        if attn_mask is not None and attn_mask.shape != (
            in_idx.shape[0],
            end,
        ):
            raise ValueError(
                "attn_mask must cover the entire cached sequence"
            )
        x = self.model.embed_tokens(in_idx)
        positions = torch.arange(
            start, end, device=in_idx.device
        ).unsqueeze(0)
        if attn_mask is not None:
            positions = (attn_mask.long().cumsum(-1) - 1).clamp(min=0)[
                :, start:end
            ]
        for i, layer in enumerate(self.model.layers):
            state = cache.get(i) if cache is not None else None
            x, state = layer(x, positions, start, state, attn_mask)
            if cache is not None:
                cache.update(i, state)
        if cache is not None:
            cache.position = end
        return self.lm_head(self.model.embedding_norm(x))

## Load the pretrained candidate

Start from the base checkpoint before running the GRPO or distillation chapters.

In [ ]:
from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.lfm3 import load_model_and_tokenizer

model, tokenizer = load_model_and_tokenizer(device=get_device())